# Quantum Digital Signature, Rebuilt for Beginners

A step-by-step teaching notebook based on the implementation in this repository.


## Purpose

This notebook is a new, beginner-friendly companion to the repository's original notebook.

It has one central goal: help a first-year college student understand **what the code is doing**, **what the mathematics means**, and **why the protocol works at all**.

We will move in a very patient order:

1. Start with digital signatures in plain language.
2. Review the quantum ideas we need from scratch.
3. Build the quantum digital signature protocol at the story level.
4. Derive the key equations one step at a time.
5. Connect every important equation to the repository's code.
6. Run the full implementation and interpret the outputs carefully.


## What This Notebook Assumes

You do **not** need prior background in:

- Dirac notation
- quantum gates
- the SWAP test
- the Holevo bound
- digital signature protocols

You only need:

- basic algebra
- comfort with binary numbers
- willingness to go slowly

Whenever a symbol appears, we will define it nearby.


## How To Use This Notebook

Read it from top to bottom and run each cell in order.

The notebook is self-contained, but it is also faithful to this repository:

- the original README was studied
- the original notebook was read fully
- the same core class and protocol flow are preserved
- the original notebook remains untouched

> Teaching promise:
>
> If a formula appears, we will explain where it comes from before we use it.


---
## 1. Why Do We Care About Digital Signatures?


### Digital Signature in Plain Language

A digital signature is a way for a sender to attach proof that:

- the message really came from them
- the message was not changed afterward

In ordinary life, a handwritten signature helps with authenticity.
In computing, we need a mathematical version of the same idea.


### Why Classical Digital Signatures Matter

Classical digital signatures are used everywhere:

- secure software updates
- banking and payments
- signed contracts
- identity verification
- secure web connections

Usually, classical schemes depend on a hard mathematical problem.
For example, the security might rely on the assumption that some computation is infeasible for an attacker.


### Why Quantum Digital Signatures Are Interesting

A quantum digital signature tries to get security from **physics** rather than from a merely hard computation.

Two important quantum ideas appear in the repository:

- **quantum states cannot be copied perfectly** if they are unknown
- **quantum measurements cannot reveal unlimited classical information**

That is why this subject is interesting: instead of saying "forging is hard," we aim to say "forging is limited by the structure of quantum theory itself."


### The Specific Problem Solved Here

This repository implements a teaching version of the **Gottesman-Chuang quantum digital signature scheme**.

The idea is:

- Alice secretly chooses classical bit strings
- each secret string is converted into a quantum state
- those quantum states act like public keys
- later, Alice reveals the appropriate secret strings to sign a message
- Bob or Charlie check whether the revealed strings match the stored quantum public keys


---
## 2. Quantum Prerequisites in Plain Language


### Bits vs Qubits

A **classical bit** can be only one of two values:

$$
0 \quad \text{or} \quad 1.
$$

A **qubit** is different.

A qubit can be in a state that combines the two basis possibilities:

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle.
$$

Here:

- $|0\rangle$ and $|1\rangle$ are the basic reference states
- $\alpha$ and $\beta$ are called **amplitudes**


### The Basis States $|0\rangle$ and $|1\rangle$

In this notebook, the two basic one-qubit states are written as

$$
|0\rangle = \begin{pmatrix}1 \\ 0\end{pmatrix},
\qquad
|1\rangle = \begin{pmatrix}0 \\ 1\end{pmatrix}.
$$

These are called the **computational basis states**.

A measurement in the computational basis asks:

- "Do I see $0$?"
- or "Do I see $1$?"


### Superposition

The expression

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle
$$

is called a **superposition**.

It does **not** mean "the qubit is secretly either $0$ or $1$ and we just do not know which."
It means the state genuinely carries both components at once until measurement.


### Probability Amplitudes

The numbers $\alpha$ and $\beta$ are not probabilities directly.
They are **probability amplitudes**.

For a valid one-qubit state, they must satisfy

$$
|\alpha|^2 + |\beta|^2 = 1.
$$

The squares of their magnitudes become probabilities.


### Measurement Probabilities

If

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
$$

then a computational-basis measurement gives

$$
P(0) = |\alpha|^2,
\qquad
P(1) = |\beta|^2.
$$

In the state family used in this repository, the amplitudes are real numbers, so we will often write

$$
P(0) = \alpha^2,
\qquad
P(1) = \beta^2.
$$


### A Very Short Introduction to Bra-Ket Notation

Dirac notation uses:

- a **ket** $|\psi\rangle$ for a column vector
- a **bra** $\langle \psi|$ for the corresponding row vector with complex conjugation

For real amplitudes, the conjugation step does nothing, so

$$
|\psi\rangle =
\begin{pmatrix}
\alpha \\
\beta
\end{pmatrix}
\quad \Longrightarrow \quad
\langle \psi| =
\begin{pmatrix}
\alpha & \beta
\end{pmatrix}.
$$

This notation becomes very convenient when we compute overlaps such as $\langle a|b\rangle$.


### Matrix-Vector Viewpoint

Quantum gates are matrices.
Quantum states are vectors.

So a gate acting on a state is just matrix multiplication.

If a gate is called $U$, then

$$
|\psi_{\text{new}}\rangle = U |\psi_{\text{old}}\rangle.
$$

That simple sentence is enough to understand a large fraction of this notebook.


### One More Helpful Fact for This Notebook

The repository uses the gate $R_Y(\phi)$, a rotation around the $y$ axis.

Because the notebook starts from $|0\rangle$ and uses only $R_Y$ in the key-state construction, the resulting amplitudes stay real.

That is helpful for beginners because the states look like

$$
\cos(\cdot)\,|0\rangle + \sin(\cdot)\,|1\rangle,
$$

not a more complicated complex-amplitude expression.


### Mini Quiz 1

Try to answer before scrolling:

1. If a qubit is in $|0\rangle$, what is the probability of measuring $1$?
2. If a qubit is in $\frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle$, what is $P(0)$?
3. Why are amplitudes not the same thing as probabilities?

Short answers:

1. $0$
2. $\frac{1}{2}$
3. Because probabilities come from squared magnitudes of amplitudes.


---
## 3. Imports and Small Helper Tools


The next cell imports the same main libraries used by the repository:

- `numpy` for numerical work
- `matplotlib` for plots
- `qiskit` and `qiskit-aer` for quantum circuits and simulation

I also define a few tiny helper functions so the later cells can focus on ideas instead of string formatting.


In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from typing import Dict, List, Tuple

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
np.set_printoptions(suppress=True)

print("=" * 60)
print("Library versions")
print("=" * 60)
import qiskit, qiskit_aer, scipy
print(f"qiskit      : {qiskit.__version__}")
print(f"qiskit-aer  : {qiskit_aer.__version__}")
print(f"numpy       : {np.__version__}")
print(f"scipy       : {scipy.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")
print("=" * 60)


In [ ]:
def bit_array_to_string(bits) -> str:
    return "".join(str(int(b)) for b in bits)


def bit_string_to_int(bit_string: str) -> int:
    return int(bit_string, 2)


def amplitudes_from_j(j: int, theta: float) -> Tuple[float, float]:
    alpha = float(np.cos(j * theta))
    beta = float(np.sin(j * theta))
    return alpha, beta


def amplitudes_from_key(key_bits, theta: float) -> Tuple[str, int, float, float]:
    if isinstance(key_bits, str):
        bit_string = key_bits
    else:
        bit_string = bit_array_to_string(key_bits)
    j = bit_string_to_int(bit_string)
    alpha, beta = amplitudes_from_j(j, theta)
    return bit_string, j, alpha, beta


def format_state(alpha: float, beta: float, decimals: int = 3) -> str:
    return f"{alpha:+.{decimals}f}|0> {beta:+.{decimals}f}|1>"


def overlap_from_indices(j_a: int, j_b: int, theta: float) -> float:
    return float(np.cos((j_a - j_b) * theta))


def swap_probability_from_overlap(overlap: float) -> float:
    return float((1.0 + abs(overlap) ** 2) / 2.0)


print("Helper functions ready.")


---
## 4. The Protocol at the Big-Picture Level


### Who Are the Main Characters?

The usual story uses three people:

- **Alice**: the signer
- **Bob**: a recipient
- **Charlie**: another recipient or judge

Alice wants to send a signed message.
Bob wants to know whether it is genuine.
Charlie matters because a useful signature should still make sense when Bob forwards it.


### The Classical One-Time Signature Idea Behind the Quantum Version

Before adding any quantum mechanics, there is a simple classical pattern:

- for each possible message bit value, Alice prepares a secret
- the corresponding public information is shared ahead of time
- later, to sign bit $0$, Alice reveals the secret for bit $0$
- to sign bit $1$, Alice reveals the secret for bit $1$

The quantum protocol in this repository follows that same spirit.


### The Story Version of This Repository's Protocol

For each message position:

1. Alice chooses two secret classical keys, one meant for signing `0` and one for signing `1`.
2. She converts each classical key into a quantum state.
3. Those quantum states play the role of public keys.
4. Recipients compare public-key copies using the SWAP test.
5. To sign the actual message, Alice reveals the matching classical keys.
6. The recipient rebuilds the expected quantum state from each revealed key and checks whether it matches the stored public key.


### The Phases We Will See in Code

The implementation naturally breaks into five phases:

- key generation
- public-key consistency checking
- signature generation
- signature verification
- security-parameter exploration


### Notation Dictionary

To keep symbols from becoming confusing, I will use:

- $L$: number of bits in one private key
- $M$: number of key pairs per message position
- $n$: number of qubits in one public-key state
- $t$: message position index
- $i$: copy index
- $k_0^{(t,i)}$: secret key used if message bit at position $t$ is $0$
- $k_1^{(t,i)}$: secret key used if message bit at position $t$ is $1$

In the code, the message-position index is called `idx_b`.


### Mini Quiz 2

Suppose Alice wants to sign the bit `1`.

Which secret should she reveal?

- the secret associated with bit `1`
- not the secret associated with bit `0`

That simple rule becomes the heart of the signing step.


---
## 5. Mathematical Foundations


### Step 1: Turn a Bit String Into an Integer

A private key is a classical bit string of length $L$:

$$
k = k_{L-1}k_{L-2}\cdots k_1k_0,
\qquad
k_r \in \{0,1\}.
$$

The repository converts that bit string into an integer

$$
j = \operatorname{int}(k, 2).
$$

The `2` means "interpret this string in base 2."


### Where Does That Integer Formula Come From?

If

$$
k = b_{L-1}b_{L-2}\cdots b_1b_0,
$$

then its value in binary is

$$
j = b_{L-1}2^{L-1} + b_{L-2}2^{L-2} + \cdots + b_1 2^1 + b_0 2^0.
$$

Example:

$$
1011_2 = 1\cdot 2^3 + 0\cdot 2^2 + 1\cdot 2^1 + 1\cdot 2^0 = 8 + 0 + 2 + 1 = 11.
$$

That integer $j$ will decide how far we rotate the qubit.


In [ ]:
bit_string_examples = ["00", "01", "10", "11", "1011", "101100"]

print("Binary string  ->  integer j")
print("-" * 32)
for bit_string in bit_string_examples:
    print(f"{bit_string:>10}  ->  {bit_string_to_int(bit_string):>3d}")


### Step 2: Why Is $\theta = \pi / 2^L$?

Once a key becomes an integer $j$, we want to map it to an angle.

There are $2^L$ possible keys of length $L$.
So it is natural to split an angular interval into $2^L$ equally spaced pieces.

The repository chooses

$$
\theta = \frac{\pi}{2^L}.
$$

Then the integer $j$ gets mapped to the angle

$$
j\theta.
$$


### Why This Choice Is Reasonable

The possible values of $j$ are

$$
0,1,2,\dots,2^L-1.
$$

So the possible angles are

$$
0,\theta,2\theta,\dots,(2^L-1)\theta.
$$

Because

$$
(2^L - 1)\theta = (2^L - 1)\frac{\pi}{2^L} = \pi - \frac{\pi}{2^L},
$$

the states spread across an interval that is almost the whole range from $0$ to $\pi$.
That gives many nearby but not identical states.


In [ ]:
print("L   number of keys   theta = pi/2^L      delta = cos(theta)")
print("-" * 62)
for L_value in range(2, 8):
    theta_value = np.pi / (2 ** L_value)
    delta_value = np.cos(theta_value)
    print(
        f"{L_value:1d}   {2**L_value:13d}   "
        f"{theta_value:14.6f}   {delta_value:16.6f}"
    )


### Step 3: The Matrix Form of the $R_Y$ Gate

The repository uses a single-qubit rotation gate:

$$
R_Y(\phi).
$$

Its matrix is

$$
R_Y(\phi)=
\begin{pmatrix}
\cos(\phi/2) & -\sin(\phi/2) \\
\sin(\phi/2) & \cos(\phi/2)
\end{pmatrix}.
$$

Here $\phi$ is the rotation angle supplied to the gate.


### Derivation: Apply $R_Y(\phi)$ to $|0\rangle$

Start from

$$
|0\rangle = \begin{pmatrix}1 \\ 0\end{pmatrix}.
$$

Then

$$
R_Y(\phi)|0\rangle
=
\begin{pmatrix}
\cos(\phi/2) & -\sin(\phi/2) \\
\sin(\phi/2) & \cos(\phi/2)
\end{pmatrix}
\begin{pmatrix}
1 \\
0
\end{pmatrix}.
$$

Multiply the matrix by the vector row by row.


### Finish the Multiplication Carefully

The top entry becomes

$$
\cos(\phi/2)\cdot 1 + \bigl(-\sin(\phi/2)\bigr)\cdot 0 = \cos(\phi/2).
$$

The bottom entry becomes

$$
\sin(\phi/2)\cdot 1 + \cos(\phi/2)\cdot 0 = \sin(\phi/2).
$$

Therefore

$$
R_Y(\phi)|0\rangle
=
\begin{pmatrix}
\cos(\phi/2) \\
\sin(\phi/2)
\end{pmatrix}
=
\cos(\phi/2)|0\rangle + \sin(\phi/2)|1\rangle.
$$


### Now Set $\phi = 2j\theta$

The repository chooses the gate angle

$$
\phi = 2j\theta.
$$

Substitute this into the formula above:

$$
R_Y(2j\theta)|0\rangle
= \cos\!\left(\frac{2j\theta}{2}\right)|0\rangle
+ \sin\!\left(\frac{2j\theta}{2}\right)|1\rangle.
$$

Since $\frac{2j\theta}{2}=j\theta$, we get

$$
|f_k\rangle
= R_Y(2j\theta)|0\rangle
= \cos(j\theta)|0\rangle + \sin(j\theta)|1\rangle.
$$

This is the key state family used by the repository.


### Plain-Language Interpretation

The private key $k$ does not directly become a probability.
It becomes an integer $j$, and that integer picks a rotation angle.

So the private key chooses a point on a smooth family of quantum states.

Nearby keys produce nearby angles.
Nearby angles produce nearby quantum states.
That is exactly why distinguishing keys is not trivial.
